# LM Studio 四模型批量 CNC test 评测

这个 notebook 与 `main_pipeline_demo1.ipynb` 隔离，复用同一套 prompt、RAG、generation 和 evaluation 代码。

运行分为六个独立阶段：

1. **Smoke 批量切换**：逐个加载模型，使用 `Heavy rain caused flooding in the region.` 调用 LM Studio 原生 API，确认 `reasoning=off`、reasoning token 为 0、没有 `<think>`，并能解析出正类 causal JSON；每个模型测试后立即卸载。即使一个模型失败，也继续测试其余模型。
2. **正式批量评测**：只有手动设置 `RUN_BATCH_EVAL=True` 才会开始。逐个加载模型、运行固定 `cnc_sft_test`、保存独立报告并卸载。该阶段既有四模型结果已经完成，开关现保持关闭。
3. **Qwen 家族 RAG3 补跑**：将在 Qwen 27B validation 消融中选出的 `k=3` 家族参数应用到 Qwen 35B A3B 和 Qwen 27B No Thinking 的完整 test，不重跑 Gemma。原运行中 35B 已正常完成，27B 中途出现异常。
4. **Qwen 27B reload100 历史补跑**：曾只重跑 Qwen3.6 27B No Thinking，现已终止并保持关闭。
5. **cache_prompt 定点诊断**：关闭 prompt cache，仅运行固定 test 第 590–636 条以复现已知异常区间。
6. **Qwen 27B 正式续跑**：复用 progress 500/1038 保存的原始计数，只对第 501–1038 条关闭 prompt cache 后补跑，再按 TP/FP/FN/TN 合并完整 test 指标。

CNC RAG 使用更新后的 train-only support；每次在固定 SFT test 上启用前都会通过 manifest 和精确 sample ID 检查验证 test overlap 为 0。


In [1]:
# ===== 全局配置：运行前先检查本单元格 =====
import json
import logging
import sys
from collections.abc import Iterable, Iterator
from html import escape
from pathlib import Path
from typing import Any

import pandas as pd
from IPython.display import HTML, display

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

if 'master_thesis' not in sys.executable.lower():
    raise RuntimeError('请切换到 Master_thesis kernel，重启 kernel 后从第一格重新运行。')

from src.data_io import load_dataset
from src.eval_pipeline import EvalRunConfig, run_stream_eval
from src.generator import generate, parse_output
from src.llm_client import LLMClient
from src.retriever import resolve_rag_cache_paths

logging.basicConfig(level=logging.INFO, format='%(asctime)s | %(levelname)s | %(message)s')
LOGGER = logging.getLogger('lmstudio_batch_cnc_test')
logging.getLogger('src.llm_client').setLevel(logging.WARNING)


def _notebook_progress(
    iterable: Iterable[Any],
    *,
    total: int,
    desc: str,
) -> Iterator[Any]:
    safe_desc = escape(desc)
    safe_total = max(total, 1)

    def render(completed: int) -> HTML:
        percent = min(completed / safe_total * 100, 100.0)
        return HTML(
            f"<div style='width:100%'>"
            f"<div>{safe_desc}: {completed}/{total} ({percent:.1f}%)</div>"
            f"<progress value='{completed}' max='{safe_total}' "
            f"style='width:100%;height:18px'></progress></div>"
        )

    handle = display(render(0), display_id=True)
    for completed, item in enumerate(iterable, 1):
        yield item
        if handle is not None:
            handle.update(render(completed))

LMSTUDIO_BASE_URL = 'http://127.0.0.1:1234/v1'
LMSTUDIO_API_KEY = 'lm-studio'
MODEL_LOAD_TIMEOUT = 1200
LLM_TIMEOUT = 600
LLM_RETRY_TIMES = 3
CONTEXT_LENGTH = 8192
MAX_TOKENS = 2048
TEMPERATURE = 0.0

DATASET_NAME = 'cnc_sft_test'
EVAL_SAMPLE_N = None  # None = 固定 test 全部 1038 条；调试时可临时设为较小整数。
EVAL_PROGRESS_EVERY = 100
EVAL_MAX_WORKERS = 1
REPORT_DIR = Path('results') / 'eval_report' / 'lmstudio_batch_base_test'
RUN_BATCH_EVAL = False  # 四模型既有 test 已完成；不要为本次 Qwen RAG3 补跑重新开启。
REQUIRE_ALL_SMOKE_PASS = True

# 四个模型分别保留自己的 prompt/RAG 配置。当前主比较与微调模型一致，统一 RAG off。
# Gemma 26B 尚无混合正负 CNC 全集的独立最佳 prompt，因此先沿用 Gemma 系列的 v9.8。
MODEL_RUNS: list[dict[str, Any]] = [
    {
        'display_name': 'Qwen3.6 35B A3B',
        'model_key': 'qwen/qwen3.6-35b-a3b',
        'prompt_name': 'v9.6',
        'use_rag': True,
        'rag_database': 'cnc',
        'rag_mode': 'knn_pattern',
        'rag_top_k': 1,
    },
    {
        'display_name': 'Qwen3.6 27B No Thinking',
        'model_key': 'local/qwen3.6-27b-no-thinking',
        'prompt_name': 'v9.6',
        'use_rag': True,
        'rag_database': 'cnc',
        'rag_mode': 'knn_pattern',
        'rag_top_k': 1,
    },
    {
        'display_name': 'Gemma 4 26B A4B QAT',
        'model_key': 'google/gemma-4-26b-a4b-qat',
        'prompt_name': 'v9.8',
        'use_rag': True,
        'rag_database': 'cnc',
        'rag_mode': 'knn_pattern',
        'rag_top_k': 1,
    },
    {
        'display_name': 'Gemma 4 31B QAT',
        'model_key': 'google/gemma-4-31b-qat',
        'prompt_name': 'v9.8',
        'use_rag': True,
        'rag_database': 'cnc',
        'rag_mode': 'knn_pattern',
        'rag_top_k': 1,
    },
]

display(pd.DataFrame(MODEL_RUNS))


,display_name,model_key,prompt_name,use_rag,rag_database,rag_mode,rag_top_k
0,Qwen3.6 35B A3B,qwen/qwen3.6-35b-a3b,v9.6,True,cnc,knn_pattern,1
1,Qwen3.6 27B No Thinking,local/qwen3.6-27b-no-thinking,v9.6,True,cnc,knn_pattern,1
2,Gemma 4 26B A4B QAT,google/gemma-4-26b-a4b-qat,v9.8,True,cnc,knn_pattern,1
3,Gemma 4 31B QAT,google/gemma-4-31b-qat,v9.8,True,cnc,knn_pattern,1


In [2]:
# ===== LM Studio 管理 API 与四个 model key 预检（不加载模型） =====
manager = LLMClient(
    provider='lmstudio',
    base_url=LMSTUDIO_BASE_URL,
    model=MODEL_RUNS[0]['model_key'],
    api_key=LMSTUDIO_API_KEY,
    temperature=TEMPERATURE,
    max_tokens=MAX_TOKENS,
    context_length=CONTEXT_LENGTH,
    reasoning='off',
    timeout=MODEL_LOAD_TIMEOUT,
    retry_times=LLM_RETRY_TIMES,
)

inventory = manager.list_local_models()
inventory_by_key = {str(item['key']): item for item in inventory if item.get('key')}
target_keys = [str(run['model_key']) for run in MODEL_RUNS]
missing_keys = [key for key in target_keys if key not in inventory_by_key]
if missing_keys:
    raise RuntimeError(f'LM Studio 中找不到目标模型：{missing_keys}')

inventory_rows: list[dict[str, Any]] = []
for run in MODEL_RUNS:
    item = inventory_by_key[str(run['model_key'])]
    reasoning = item.get('capabilities', {}).get('reasoning', {})
    allowed = reasoning.get('allowed_options', [])
    inventory_rows.append({
        'model': run['display_name'],
        'model_key': run['model_key'],
        'quantization': item.get('quantization', {}).get('name'),
        'params': item.get('params_string'),
        'reasoning_default': reasoning.get('default'),
        'reasoning_off_capability': (
            'advertised' if 'off' in allowed
            else 'not_supported' if allowed
            else 'smoke_required'
        ),
        'loaded_instances': len(item.get('loaded_instances', [])),
    })

inventory_df = pd.DataFrame(inventory_rows)
display(inventory_df)
unsupported = inventory_df.loc[
    inventory_df['reasoning_off_capability'] == 'not_supported', 'model_key'
].tolist()
if unsupported:
    raise RuntimeError(f'以下模型不支持 reasoning=off：{unsupported}')
LOGGER.info('LM Studio 管理 API 与四个目标 model key 检查通过；smoke_required 将由实际生成验证。')


,model,model_key,quantization,params,reasoning_default,reasoning_off_capability,loaded_instances
0,Qwen3.6 35B A3B,qwen/qwen3.6-35b-a3b,Q4_K_M,35B-A3B,off,advertised,0
1,Qwen3.6 27B No Thinking,local/qwen3.6-27b-no-thinking,Q4_K_M,27B,off,advertised,0
2,Gemma 4 26B A4B QAT,google/gemma-4-26b-a4b-qat,Q4_0,26B-A4B,off,advertised,0
3,Gemma 4 31B QAT,google/gemma-4-31b-qat,Q4_0,31B,off,advertised,0


2026-08-26 16:28:14,424 | INFO | LM Studio 管理 API 与四个目标 model key 检查通过；smoke_required 将由实际生成验证。


In [3]:
# ===== Smoke 批量切换辅助函数 =====
SMOKE_TEXT = 'Heavy rain caused flooding in the region.'
SMOKE_SYSTEM_PROMPT = (
    'Extract causal relations and output one strict JSON object only. '
    'Use schema {\"has_causal\": true, \"triples\": '    '[{\"cause\": {\"span\": \"...\"}, \"relation\": \"caused\", '    '\"effect\": {\"span\": \"...\"}}]}. '    'Do not output reasoning, Markdown, or any text outside the JSON.'
)


def _message_text(payload: dict[str, Any]) -> str:
    return '\n'.join(
        str(item.get('content', ''))
        for item in payload.get('output', [])
        if isinstance(item, dict) and item.get('type') == 'message'
    ).strip()


def run_smoke_batch(
    model_runs: list[dict[str, Any]],
    lmstudio: LLMClient,
) -> list[dict[str, Any]]:
    results: list[dict[str, Any]] = []
    for position, run in enumerate(model_runs, 1):
        row: dict[str, Any] = {
            'order': position,
            'model': run['display_name'],
            'model_key': run['model_key'],
            'load': False,
            'reasoning_tokens': None,
            'no_reasoning': False,
            'causal_json': False,
            'unload': False,
            'passed': False,
            'error': '',
        }
        LOGGER.info('[smoke %s/%s] 准备加载 %s', position, len(model_runs), run['model_key'])
        try:
            lmstudio.unload_all_models()
            load_result = lmstudio.load_model(
                str(run['model_key']),
                context_length=CONTEXT_LENGTH,
            )
            instance_id = str(load_result['instance_id'])
            row['instance_id'] = instance_id
            row['load_config'] = json.dumps(load_result.get('load_config', {}), ensure_ascii=False)
            row['load'] = load_result.get('status') == 'loaded'

            smoke_payload = lmstudio.chat_rest(
                SMOKE_TEXT,
                model=instance_id,
                system_prompt=SMOKE_SYSTEM_PROMPT,
                reasoning='off',
                max_output_tokens=512,
            )
            content = _message_text(smoke_payload)
            reasoning_items = [
                item
                for item in smoke_payload.get('output', [])
                if isinstance(item, dict) and item.get('type') == 'reasoning'
            ]
            reasoning_tokens = smoke_payload.get('stats', {}).get('reasoning_output_tokens')
            row['reasoning_tokens'] = reasoning_tokens
            row['no_reasoning'] = (
                reasoning_tokens == 0
                and not reasoning_items
                and '<think>' not in content.lower()
            )
            parsed = parse_output(content)
            row['causal_json'] = bool(parsed.get('has_causal')) and bool(parsed.get('triples'))
            row['raw_output'] = content
            row['passed'] = bool(row['load'] and row['no_reasoning'] and row['causal_json'])
        except Exception as exc:
            row['error'] = f'{type(exc).__name__}: {exc}'
            LOGGER.exception('[smoke %s/%s] %s 失败', position, len(model_runs), run['model_key'])
        finally:
            try:
                lmstudio.unload_all_models()
                row['unload'] = True
            except Exception as unload_exc:
                row['passed'] = False
                unload_message = f'{type(unload_exc).__name__}: {unload_exc}'
                row['error'] = '; '.join(part for part in [row['error'], unload_message] if part)
                LOGGER.exception('[smoke %s/%s] 卸载失败', position, len(model_runs))
        results.append(row)
    return results


In [4]:
# ===== 阶段 1：四模型逐个 load -> smoke -> unload =====
SMOKE_RESULTS = run_smoke_batch(MODEL_RUNS, manager)
smoke_df = pd.DataFrame(SMOKE_RESULTS)
display(smoke_df[[
    'order', 'model', 'model_key', 'load', 'reasoning_tokens',
    'no_reasoning', 'causal_json', 'unload', 'passed', 'error',
]])
LOGGER.info('Smoke 通过：%s/%s', int(smoke_df['passed'].sum()), len(smoke_df))


2026-08-26 16:28:35,905 | INFO | [smoke 1/4] 准备加载 qwen/qwen3.6-35b-a3b
2026-08-26 16:28:57,871 | INFO | [smoke 2/4] 准备加载 local/qwen3.6-27b-no-thinking
2026-08-26 16:29:16,030 | INFO | [smoke 3/4] 准备加载 google/gemma-4-26b-a4b-qat
2026-08-26 16:29:30,840 | INFO | [smoke 4/4] 准备加载 google/gemma-4-31b-qat


,order,model,model_key,load,reasoning_tokens,no_reasoning,causal_json,unload,passed,error
0,1,Qwen3.6 35B A3B,qwen/qwen3.6-35b-a3b,True,0,True,True,True,True,
1,2,Qwen3.6 27B No Thinking,local/qwen3.6-27b-no-thinking,True,0,True,True,True,True,
2,3,Gemma 4 26B A4B QAT,google/gemma-4-26b-a4b-qat,True,0,True,True,True,True,
3,4,Gemma 4 31B QAT,google/gemma-4-31b-qat,True,0,True,True,True,True,


2026-08-26 16:29:48,965 | INFO | Smoke 通过：4/4


## 阶段 2：正式固定 test 批量评测

先确认上面的四行 smoke 均为 `passed=True`。正式评测不会自动开始；回到全局配置，将 `RUN_BATCH_EVAL` 改为 `True`，再运行下面两个单元格。

每个模型都使用自己的配置行。正式评测使用模型定义中的 `Enable Thinking=False`；smoke 另行通过原生 API 验证 `reasoning=off`。每个模型结束后，无论成功或失败，都会进入卸载步骤。


In [5]:
# ===== 正式批量评测辅助函数 =====
def _extra_body_for_run(run: dict[str, Any]) -> dict[str, Any]:
    return {'cache_prompt': False, **(run.get('llm_extra_body') or {})}


def _rag_paths_for_run(run: dict[str, Any]) -> tuple[Path | None, Path | None]:
    if not run['use_rag']:
        return None, None

    metadata_path, embeddings_path = resolve_rag_cache_paths(str(run['rag_database']))
    if DATASET_NAME == 'cnc_sft_test' and str(run['rag_database']) == 'cnc':
        sft_manifest = json.loads((PROJECT_ROOT / 'Data/CNC_sft/split_manifest.json').read_text(encoding='utf-8'))
        rag_manifest = json.loads((PROJECT_ROOT / 'RAG Database/cnc_split_manifest.json').read_text(encoding='utf-8'))
        test_ids = {str(sample_id) for sample_id in sft_manifest['split_ids']['test']}
        overlap = test_ids.intersection(str(sample_id) for sample_id in rag_manifest['support_ids'])
        if overlap:
            raise RuntimeError(
                f'拒绝在 cnc_sft_test 上使用现有 CNC RAG：support 与 test 有 {len(overlap)} 个精确 ID 重叠。'
            )
    return metadata_path, embeddings_path


def _load_eval_client(run: dict[str, Any]) -> tuple[str, LLMClient]:
    run_context_length = int(run.get('context_length', CONTEXT_LENGTH))
    run_parallel = run.get('parallel')
    run_offload_kv = bool(run.get('offload_kv_cache_to_gpu', True))
    load_result = manager.load_model(
        str(run['model_key']),
        context_length=run_context_length,
        parallel=None if run_parallel is None else int(run_parallel),
        offload_kv_cache_to_gpu=run_offload_kv,
    )
    load_config = load_result.get('load_config', {})
    actual_context = int(load_config.get('context_length', 0))
    actual_parallel = load_config.get('parallel')
    actual_offload_kv = load_config.get('offload_kv_cache_to_gpu')
    if (
        actual_context != run_context_length
        or (run_parallel is not None and int(actual_parallel or 0) != int(run_parallel))
        or actual_offload_kv is not run_offload_kv
    ):
        raise RuntimeError(
            '模型实际加载配置与目标不一致：'
            f'context={actual_context}/{run_context_length}, '
            f'parallel={actual_parallel}/{run_parallel}, '
            f'kv_cache_gpu={actual_offload_kv}/{run_offload_kv}。'
        )
    instance_id = str(load_result['instance_id'])
    client = LLMClient(
        provider='lmstudio',
        base_url=LMSTUDIO_BASE_URL,
        model=instance_id,
        api_key=LMSTUDIO_API_KEY,
        temperature=TEMPERATURE,
        max_tokens=MAX_TOKENS,
        context_length=run_context_length,
        extra_body=_extra_body_for_run(run),
        timeout=LLM_TIMEOUT,
        retry_times=LLM_RETRY_TIMES,
    )
    return instance_id, client


def _summary_row(run: dict[str, Any], report: dict[str, Any]) -> dict[str, Any]:
    detection = report['detection']
    strict = report['extraction']['strict_token_f1']
    anchor = report['extraction']['anchor_window']
    return {
        'status': 'completed',
        'model': run['display_name'],
        'model_key': run['model_key'],
        'prompt': run['prompt_name'],
        'rag': 'off' if not run['use_rag'] else f"{run['rag_mode']}-k{run['rag_top_k']}",
        'temperature': TEMPERATURE,
        'context_length': int(run.get('context_length', CONTEXT_LENGTH)),
        'max_tokens': MAX_TOKENS,
        'parallel': run.get('parallel', 'model_default'),
        'reload_every_samples': int(run.get('reload_every_samples', 0) or 0),
        'cache_prompt': _extra_body_for_run(run)['cache_prompt'],
        'offload_kv_cache_to_gpu': bool(run.get('offload_kv_cache_to_gpu', True)),
        'reasoning': 'model_default_off',
        'n_samples': report['n_samples'],
        'n_causal_gold': report['n_causal_gold'],
        'n_causal_pred': report['n_causal_pred'],
        'detection_tp': detection['tp'],
        'detection_tn': detection['tn'],
        'detection_fp': detection['fp'],
        'detection_fn': detection['fn'],
        'all_n_eval_samples': strict['all_samples']['n_eval_samples'],
        'all_n_gold_triples': strict['all_samples']['n_gold_triples'],
        'all_n_pred_triples': strict['all_samples']['n_pred_triples'],
        'strict_all_tp': strict['all_samples']['tp'],
        'strict_all_fp': strict['all_samples']['fp'],
        'strict_all_fn': strict['all_samples']['fn'],
        'anchor_all_tp': anchor['all_samples']['tp'],
        'anchor_all_fp': anchor['all_samples']['fp'],
        'anchor_all_fn': anchor['all_samples']['fn'],
        'detected_n_eval_samples': strict['detected_only']['n_eval_samples'],
        'detected_n_gold_triples': strict['detected_only']['n_gold_triples'],
        'detected_n_pred_triples': strict['detected_only']['n_pred_triples'],
        'strict_detected_tp': strict['detected_only']['tp'],
        'strict_detected_fp': strict['detected_only']['fp'],
        'strict_detected_fn': strict['detected_only']['fn'],
        'anchor_detected_tp': anchor['detected_only']['tp'],
        'anchor_detected_fp': anchor['detected_only']['fp'],
        'anchor_detected_fn': anchor['detected_only']['fn'],
        'detection_precision': detection['precision'],
        'detection_recall': detection['recall'],
        'detection_f1': detection['f1'],
        'strict_all_precision': strict['all_samples']['precision'],
        'strict_all_recall': strict['all_samples']['recall'],
        'strict_all_f1': strict['all_samples']['f1'],
        'anchor_all_precision': anchor['all_samples']['precision'],
        'anchor_all_recall': anchor['all_samples']['recall'],
        'anchor_all_f1': anchor['all_samples']['f1'],
        'strict_detected_only_f1': strict['detected_only']['f1'],
        'anchor_detected_only_f1': anchor['detected_only']['f1'],
        'report_path': report.get('report_path', ''),
        'error': '',
    }


def run_formal_batch(
    model_runs: list[dict[str, Any]],
    *,
    existing_retrievers: dict[str, Any] | None = None,
    samples_override: list[dict[str, Any]] | None = None,
    require_all_smoke: bool = REQUIRE_ALL_SMOKE_PASS,
) -> list[dict[str, Any]]:
    if require_all_smoke:
        if 'SMOKE_RESULTS' not in globals():
            raise RuntimeError('请先运行四模型 smoke 单元格。')
        failed_smokes = [row['model_key'] for row in SMOKE_RESULTS if not row['passed']]
        if failed_smokes:
            raise RuntimeError(f'以下模型 smoke 未通过，正式评测未启动：{failed_smokes}')

    samples = (
        samples_override
        if samples_override is not None
        else load_dataset(DATASET_NAME, n=EVAL_SAMPLE_N)
    )
    rows: list[dict[str, Any]] = []
    for position, run in enumerate(model_runs, 1):
        LOGGER.info('[eval %s/%s] 开始 %s', position, len(model_runs), run['model_key'])
        try:
            manager.unload_all_models()
            metadata_path, embeddings_path = _rag_paths_for_run(run)
            run_id = str(run.get('run_id', run['model_key']))
            run_context_length = int(run.get('context_length', CONTEXT_LENGTH))
            reload_every_samples = int(run.get('reload_every_samples', 0) or 0)
            if reload_every_samples < 0:
                raise ValueError('reload_every_samples 不能为负数。')
            if reload_every_samples and EVAL_MAX_WORKERS != 1:
                raise RuntimeError('周期性模型重载要求 EVAL_MAX_WORKERS=1，以确保按样本计数。')
            instance_id, client = _load_eval_client(run)
            completed_samples = 0

            def generate_with_periodic_reload(**generator_kwargs: Any) -> dict[str, Any]:
                nonlocal instance_id, client, completed_samples
                if (
                    reload_every_samples
                    and completed_samples > 0
                    and completed_samples % reload_every_samples == 0
                ):
                    LOGGER.info(
                        '[eval] 已完成 %s 条，重新加载 %s 后继续。',
                        completed_samples,
                        run['model_key'],
                    )
                    manager.unload_all_models()
                    instance_id, client = _load_eval_client(run)
                generator_kwargs['client'] = client
                prediction = generate(**generator_kwargs)
                completed_samples += 1
                return prediction
            eval_config = EvalRunConfig(
                project_root=PROJECT_ROOT,
                model=instance_id,
                dataset=DATASET_NAME,
                prompt_name=str(run['prompt_name']),
                use_rag=bool(run['use_rag']),
                rag_mode=str(run['rag_mode']),
                rag_top_k=int(run['rag_top_k']),
                temperature=TEMPERATURE,
                max_tokens=MAX_TOKENS,
                primary_metric=str(run.get('primary_metric', 'anchor_window')),
                progress_every=EVAL_PROGRESS_EVERY,
                max_workers=EVAL_MAX_WORKERS,
                llm_provider='lmstudio',
                llm_base_url=LMSTUDIO_BASE_URL,
                context_length=run_context_length,
                reasoning_effort=None,
                llm_extra_body=_extra_body_for_run(run),
                api_key_source='lmstudio-default',
                save_report=True,
                report_dir=REPORT_DIR,
                report_detail_limit=200,
                report_detail_mode='errors',
                report_error_metric=str(run.get('primary_metric', 'anchor_window')),
                metadata_path=metadata_path,
                embeddings_path=embeddings_path,
            )
            report = run_stream_eval(
                samples=samples,
                label=str(run.get('label', f"{run['display_name']} CNC fixed test")),
                client=client,
                config=eval_config,
                generator=generate_with_periodic_reload,
                existing_retriever=(existing_retrievers or {}).get(run_id),
                progress_factory=_notebook_progress,
                emit=LOGGER.info,
            )
            rows.append(_summary_row(run, report))
        except Exception as exc:
            LOGGER.exception('[eval %s/%s] %s 失败', position, len(model_runs), run['model_key'])
            rows.append({
                'status': 'failed',
                'model': run['display_name'],
                'model_key': run['model_key'],
                'prompt': run['prompt_name'],
                'rag': 'off' if not run['use_rag'] else f"{run['rag_mode']}-k{run['rag_top_k']}",
                'temperature': TEMPERATURE,
                'context_length': CONTEXT_LENGTH,
                'max_tokens': MAX_TOKENS,
                'parallel': run.get('parallel', 'model_default'),
                'reload_every_samples': int(run.get('reload_every_samples', 0) or 0),
                'cache_prompt': _extra_body_for_run(run)['cache_prompt'],
                'reasoning': 'model_default_off',
                'error': f'{type(exc).__name__}: {exc}',
            })
        finally:
            try:
                manager.unload_all_models()
            except Exception:
                LOGGER.exception('[eval %s/%s] 评测后卸载失败', position, len(model_runs))
    return rows


In [6]:
# ===== 阶段 2：正式批量评测（默认不会运行） =====
if not RUN_BATCH_EVAL:
    LOGGER.warning('正式批量评测未启动。确认 smoke 后，将 RUN_BATCH_EVAL 改为 True。')
else:
    BATCH_RESULTS = run_formal_batch(MODEL_RUNS)
    batch_summary_df = pd.DataFrame(BATCH_RESULTS)
    display(batch_summary_df)


2026-08-26 16:30:19,137 | INFO | [eval 1/4] 开始 qwen/qwen3.6-35b-a3b
2026-08-26 16:30:37,720 | INFO | Pattern examples 已加载：path=D:\Master thesis\RAG Database\cnc_examples.jsonl examples=600
2026-08-26 16:30:37,733 | INFO | KNN cache 已加载：examples=600 path=D:\Master thesis\RAG Database\cnc_examples.jsonl embedding_device=cpu


D:\Anaconda3\envs\Master_thesis\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2026-08-26 16:30:49,693 | INFO | HTTP Request: HEAD https://huggingface.co/BAAI/bge-small-en-v1.5/resolve/main/modules.json "HTTP/1.1 307 Temporary Redirect"
2026-08-26 16:30:49,724 | INFO | HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/BAAI/bge-small-en-v1.5/5c38ec7c405ec4b44b94cc5a9bb96e735b38267a/modules.json "HTTP/1.1 200 OK"
2026-08-26 16:30:49,880 | INFO | HTTP Request: HEAD https://huggingface.co/BAAI/bge-small-en-v1.5/resolve/main/config_sentence_transformers.json "HTTP/1.1 307 Temporary Redirect"
2026-08-26 16:30:49,910 | INFO | HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/BAAI/bge-small-en-v1.5/5c38ec7c405ec4b44b94cc5a9bb96e735b38267a/config_sentence_transformers.json "HTTP/1.1 200 O

2026-08-26 16:31:35,053 | INFO | HTTP Request: POST http://127.0.0.1:1234/v1/chat/completions "HTTP/1.1 200 OK"
2026-08-26 16:31:35,947 | INFO | HTTP Request: POST http://127.0.0.1:1234/v1/chat/completions "HTTP/1.1 200 OK"
2026-08-26 16:31:36,953 | INFO | HTTP Request: POST http://127.0.0.1:1234/v1/chat/completions "HTTP/1.1 200 OK"
2026-08-26 16:31:37,787 | INFO | HTTP Request: POST http://127.0.0.1:1234/v1/chat/completions "HTTP/1.1 200 OK"
2026-08-26 16:31:38,774 | INFO | HTTP Request: POST http://127.0.0.1:1234/v1/chat/completions "HTTP/1.1 200 OK"
2026-08-26 16:31:39,670 | INFO | HTTP Request: POST http://127.0.0.1:1234/v1/chat/completions "HTTP/1.1 200 OK"
2026-08-26 16:31:40,581 | INFO | HTTP Request: POST http://127.0.0.1:1234/v1/chat/completions "HTTP/1.1 200 OK"
2026-08-26 16:31:42,864 | INFO | HTTP Request: POST http://127.0.0.1:1234/v1/chat/completions "HTTP/1.1 200 OK"
2026-08-26 16:31:43,917 | INFO | HTTP Request: POST http://127.0.0.1:1234/v1/chat/completions "HTTP/1.1 

2026-08-26 16:33:46,065 | INFO | HTTP Request: POST http://127.0.0.1:1234/v1/chat/completions "HTTP/1.1 200 OK"
2026-08-26 16:33:48,380 | INFO | HTTP Request: POST http://127.0.0.1:1234/v1/chat/completions "HTTP/1.1 200 OK"
2026-08-26 16:33:49,365 | INFO | HTTP Request: POST http://127.0.0.1:1234/v1/chat/completions "HTTP/1.1 200 OK"
2026-08-26 16:33:52,235 | INFO | HTTP Request: POST http://127.0.0.1:1234/v1/chat/completions "HTTP/1.1 200 OK"
2026-08-26 16:33:54,426 | INFO | HTTP Request: POST http://127.0.0.1:1234/v1/chat/completions "HTTP/1.1 200 OK"
2026-08-26 16:33:56,415 | INFO | HTTP Request: POST http://127.0.0.1:1234/v1/chat/completions "HTTP/1.1 200 OK"
2026-08-26 16:33:58,939 | INFO | HTTP Request: POST http://127.0.0.1:1234/v1/chat/completions "HTTP/1.1 200 OK"
2026-08-26 16:33:58,959 | INFO | 
2026-08-26 16:33:58,960 | INFO | ================ Qwen3.6 35B A3B CNC fixed test progress 100/1038 ================
样本总数: 100
  Gold 含因果: 57 | Pred 含因果: 56
  Primary extraction metri

2026-08-26 16:35:30,985 | INFO | HTTP Request: POST http://127.0.0.1:1234/v1/chat/completions "HTTP/1.1 200 OK"
2026-08-26 16:35:34,075 | INFO | HTTP Request: POST http://127.0.0.1:1234/v1/chat/completions "HTTP/1.1 200 OK"
2026-08-26 16:35:34,989 | INFO | HTTP Request: POST http://127.0.0.1:1234/v1/chat/completions "HTTP/1.1 200 OK"
2026-08-26 16:35:35,871 | INFO | HTTP Request: POST http://127.0.0.1:1234/v1/chat/completions "HTTP/1.1 200 OK"
2026-08-26 16:35:37,710 | INFO | HTTP Request: POST http://127.0.0.1:1234/v1/chat/completions "HTTP/1.1 200 OK"
2026-08-26 16:35:40,038 | INFO | HTTP Request: POST http://127.0.0.1:1234/v1/chat/completions "HTTP/1.1 200 OK"
2026-08-26 16:35:42,044 | INFO | HTTP Request: POST http://127.0.0.1:1234/v1/chat/completions "HTTP/1.1 200 OK"
2026-08-26 16:35:43,073 | INFO | HTTP Request: POST http://127.0.0.1:1234/v1/chat/completions "HTTP/1.1 200 OK"
2026-08-26 16:35:45,787 | INFO | HTTP Request: POST http://127.0.0.1:1234/v1/chat/completions "HTTP/1.1 

...

2026-08-26 18:54:21,548 | INFO | HTTP Request: POST http://127.0.0.1:1234/v1/chat/completions "HTTP/1.1 200 OK"
2026-08-26 18:54:28,074 | INFO | HTTP Request: POST http://127.0.0.1:1234/v1/chat/completions "HTTP/1.1 200 OK"
2026-08-26 18:54:34,897 | INFO | HTTP Request: POST http://127.0.0.1:1234/v1/chat/completions "HTTP/1.1 200 OK"
2026-08-26 18:54:39,621 | INFO | HTTP Request: POST http://127.0.0.1:1234/v1/chat/completions "HTTP/1.1 200 OK"
2026-08-26 18:54:44,502 | INFO | HTTP Request: POST http://127.0.0.1:1234/v1/chat/completions "HTTP/1.1 200 OK"
2026-08-26 18:54:46,033 | INFO | HTTP Request: POST http://127.0.0.1:1234/v1/chat/completions "HTTP/1.1 200 OK"
2026-08-26 18:54:51,636 | INFO | HTTP Request: POST http://127.0.0.1:1234/v1/chat/completions "HTTP/1.1 200 OK"
2026-08-26 18:54:55,804 | INFO | HTTP Request: POST http://127.0.0.1:1234/v1/chat/completions "HTTP/1.1 200 OK"
2026-08-26 18:54:57,324 | INFO | HTTP Request: POST http://127.0.0.1:1234/v1/chat/completions "HTTP/1.1 

2026-08-26 18:58:36,892 | INFO | HTTP Request: POST http://127.0.0.1:1234/v1/chat/completions "HTTP/1.1 200 OK"
2026-08-26 18:58:38,403 | INFO | HTTP Request: POST http://127.0.0.1:1234/v1/chat/completions "HTTP/1.1 200 OK"
2026-08-26 18:58:44,897 | INFO | HTTP Request: POST http://127.0.0.1:1234/v1/chat/completions "HTTP/1.1 200 OK"
2026-08-26 18:58:50,325 | INFO | HTTP Request: POST http://127.0.0.1:1234/v1/chat/completions "HTTP/1.1 200 OK"
2026-08-26 18:58:51,826 | INFO | HTTP Request: POST http://127.0.0.1:1234/v1/chat/completions "HTTP/1.1 200 OK"
2026-08-26 18:58:53,374 | INFO | HTTP Request: POST http://127.0.0.1:1234/v1/chat/completions "HTTP/1.1 200 OK"
2026-08-26 18:58:54,894 | INFO | HTTP Request: POST http://127.0.0.1:1234/v1/chat/completions "HTTP/1.1 200 OK"
2026-08-26 18:58:59,689 | INFO | HTTP Request: POST http://127.0.0.1:1234/v1/chat/completions "HTTP/1.1 200 OK"
2026-08-26 18:59:01,200 | INFO | HTTP Request: POST http://127.0.0.1:1234/v1/chat/completions "HTTP/1.1 

2026-08-26 19:02:43,992 | INFO | HTTP Request: POST http://127.0.0.1:1234/v1/chat/completions "HTTP/1.1 200 OK"
2026-08-26 19:02:45,510 | INFO | HTTP Request: POST http://127.0.0.1:1234/v1/chat/completions "HTTP/1.1 200 OK"
2026-08-26 19:02:50,688 | INFO | HTTP Request: POST http://127.0.0.1:1234/v1/chat/completions "HTTP/1.1 200 OK"
2026-08-26 19:03:01,601 | INFO | HTTP Request: POST http://127.0.0.1:1234/v1/chat/completions "HTTP/1.1 200 OK"
2026-08-26 19:03:03,181 | INFO | HTTP Request: POST http://127.0.0.1:1234/v1/chat/completions "HTTP/1.1 200 OK"
2026-08-26 19:03:04,697 | INFO | HTTP Request: POST http://127.0.0.1:1234/v1/chat/completions "HTTP/1.1 200 OK"
2026-08-26 19:03:06,185 | INFO | HTTP Request: POST http://127.0.0.1:1234/v1/chat/completions "HTTP/1.1 200 OK"
2026-08-26 19:03:15,453 | INFO | HTTP Request: POST http://127.0.0.1:1234/v1/chat/completions "HTTP/1.1 200 OK"
2026-08-26 19:03:22,866 | INFO | HTTP Request: POST http://127.0.0.1:1234/v1/chat/completions "HTTP/1.1 

2026-08-26 19:04:26,696 | INFO | Eval report saved: D:\Master thesis\results\eval_report\lmstudio_batch_base_test\google-gemma-4-31b-qat_cnc_sft_test-n1038_prompt-v9.8_rag-knn_pattern-k1_20260826-190426.md
2026-08-26 19:04:26,697 | INFO | Parse repair outputs saved: D:\Master thesis\results\eval_report\lmstudio_batch_base_test\google-gemma-4-31b-qat_cnc_sft_test-n1038_prompt-v9.8_rag-knn_pattern-k1_20260826-190426_parse_repairs.jsonl


,status,model,model_key,prompt,rag,temperature,context_length,max_tokens,reasoning,detection_precision,...,strict_all_precision,strict_all_recall,strict_all_f1,anchor_all_precision,anchor_all_recall,anchor_all_f1,strict_detected_only_f1,anchor_detected_only_f1,report_path,error
0,completed,Qwen3.6 35B A3B,qwen/qwen3.6-35b-a3b,v9.6,knn_pattern-k1,0.0,8192,2048,model_default_off,0.801056,...,0.492980,0.414155,0.450142,0.407176,0.342071,0.371795,0.534235,0.441251,D:\Master thesis\results\eval_report\lmstudio_...,
1,completed,Qwen3.6 27B No Thinking,local/qwen3.6-27b-no-thinking,v9.6,knn_pattern-k1,0.0,8192,2048,model_default_off,0.811215,...,0.552860,0.418087,0.476119,0.578856,0.437746,0.498507,0.577376,0.604525,D:\Master thesis\results\eval_report\lmstudio_...,
2,completed,Gemma 4 26B A4B QAT,google/gemma-4-26b-a4b-qat,v9.8,knn_pattern-k1,0.0,8192,2048,model_default_off,0.741935,...,0.484355,0.507208,0.495519,0.540676,0.566186,0.553137,0.587253,0.655539,D:\Master thesis\results\eval_report\lmstudio_...,
3,completed,Gemma 4 31B QAT,google/gemma-4-31b-qat,v9.8,knn_pattern-k1,0.0,8192,2048,model_default_off,0.778120,...,0.545006,0.579292,0.561626,0.625154,0.664482,0.644219,0.644315,0.739067,D:\Master thesis\results\eval_report\lmstudio_...,


## 阶段 3：Qwen validation-selected RAG3 完整 test 补跑

Qwen3.6 27B No Thinking 作为 Qwen 家族代表模型，在 CNC validation 上由主指标 `strict_token_f1` 选出的家族配置是 **v9.6 Fixed 10-shot + KNN+Pattern RAG k=3**。本节将该参数用于 Qwen3.6 35B A3B 与 Qwen3.6 27B No Thinking 两个 Qwen 模型的完整 `cnc_sft_test`，不会重跑两个 Gemma 模型或阶段 2 的既有结果。

动态检索严格返回 3 条 Pattern 和 3 条 KNN train-only examples，因此每条请求共有 16 条 examples。两个模型均以 `context_length=8192`、`parallel=1`、KV cache GPU offload 开启的配置依次加载，并在每完成 200 条样本后卸载、重载同一模型；各自的详细报告与家族补跑 summary CSV 都会保留。


In [ ]:
# ===== Qwen RAG3 test 配置与隔离预检（不加载 LLM） =====
from src.prompt_builder import build_messages, load_prompt_template
from src.retriever import ExactCountHybridRetriever, KNNRetriever, PatternRetriever

RUN_QWEN_FAMILY_RAG3_TEST = False  # 35B 已完成、27B 已中止；不要再次运行原双模型入口。
QWEN_FAMILY_RAG3_TEST_SUMMARY_PATH = (
    PROJECT_ROOT / REPORT_DIR / 'qwen_family_fixed_rag_k3_test_summary.csv'
)
QWEN_FAMILY_RAG3_TEST_RUNS: list[dict[str, Any]] = [
    {
        'run_id': 'qwen35a3b_fixed_rag_k3_test',
        'display_name': 'Qwen3.6 35B A3B',
        'model_key': 'qwen/qwen3.6-35b-a3b',
        'prompt_name': 'v9.6',
        'use_rag': True,
        'rag_database': 'cnc',
        'rag_mode': 'knn_pattern',
        'rag_top_k': 3,
        'dynamic_examples': 6,
        'total_examples': 16,
        'primary_metric': 'strict_token_f1',
        'context_length': 8192,
        'parallel': 1,
        'reload_every_samples': 200,
        'offload_kv_cache_to_gpu': True,
        'label': 'Qwen3.6 35B A3B CNC test fixed_rag_k3',
    },
    {
        'run_id': 'qwen27_fixed_rag_k3_test',
        'display_name': 'Qwen3.6 27B No Thinking',
        'model_key': 'local/qwen3.6-27b-no-thinking',
        'prompt_name': 'v9.6',
        'use_rag': True,
        'rag_database': 'cnc',
        'rag_mode': 'knn_pattern',
        'rag_top_k': 3,
        'dynamic_examples': 6,
        'total_examples': 16,
        'primary_metric': 'strict_token_f1',
        'context_length': 8192,
        'parallel': 1,
        'reload_every_samples': 200,
        'offload_kv_cache_to_gpu': True,
        'label': 'Qwen3.6 27B No Thinking CNC test fixed_rag_k3',
    },
]

qwen_rag3_metadata_path, qwen_rag3_embeddings_path = _rag_paths_for_run(
    QWEN_FAMILY_RAG3_TEST_RUNS[0]
)
qwen_rag3_manifest = json.loads(
    (PROJECT_ROOT / 'RAG Database/cnc_split_manifest.json').read_text(encoding='utf-8')
)
if not qwen_rag3_manifest.get('train_only_support'):
    raise RuntimeError('CNC RAG manifest 未声明 train_only_support。')

qwen_rag3_template = load_prompt_template(QWEN_FAMILY_RAG3_TEST_RUNS[0]['prompt_name'])
if qwen_rag3_template.count('\nInput:\n') != 10:
    raise RuntimeError('v9.6 不包含恰好 10 个 fixed examples。')
if qwen_rag3_template.count('{rag_examples}') != 1:
    raise RuntimeError('v9.6 必须包含且只包含一个 {rag_examples} 占位符。')

QWEN_RAG3_RETRIEVER = ExactCountHybridRetriever(
    pattern_retriever=PatternRetriever(metadata_path=qwen_rag3_metadata_path),
    knn_retriever=KNNRetriever(
        metadata_path=qwen_rag3_metadata_path,
        embeddings_path=qwen_rag3_embeddings_path,
        device='cpu',
    ),
)
qwen_rag3_audit_text = 'Heavy rain caused flooding in the region.'
qwen_rag3_examples = QWEN_RAG3_RETRIEVER.retrieve(qwen_rag3_audit_text, top_k=3)
qwen_rag3_messages = build_messages(
    qwen_rag3_audit_text,
    use_rag=True,
    retriever=QWEN_RAG3_RETRIEVER,
    top_k=3,
    rag_mode='knn_pattern',
    prompt_name='v9.6',
)
inserted_examples = qwen_rag3_messages[0]['content'].count('\nExample ')
if len(qwen_rag3_examples) != 6 or inserted_examples != 6:
    raise RuntimeError(
        f'Qwen RAG3 预检未得到严格 6 条动态示例：'
        f'retrieved={len(qwen_rag3_examples)} inserted={inserted_examples}'
    )
display(pd.DataFrame(QWEN_FAMILY_RAG3_TEST_RUNS))
LOGGER.info(
    'Qwen 家族 RAG3 test 预检通过；summary 将保存到：%s',
    QWEN_FAMILY_RAG3_TEST_SUMMARY_PATH,
)


In [ ]:
# ===== 依次补跑两个 Qwen 的 Fixed + RAG k=3 完整 test =====
if not RUN_QWEN_FAMILY_RAG3_TEST:
    LOGGER.warning(
        'Qwen 家族 RAG3 test 补跑未启动。将 RUN_QWEN_FAMILY_RAG3_TEST 改为 True。'
    )
else:
    QWEN_FAMILY_RAG3_TEST_RESULTS = run_formal_batch(
        QWEN_FAMILY_RAG3_TEST_RUNS,
        existing_retrievers={
            run['run_id']: QWEN_RAG3_RETRIEVER
            for run in QWEN_FAMILY_RAG3_TEST_RUNS
        },
        require_all_smoke=False,
    )
    qwen_family_rag3_test_df = pd.DataFrame(QWEN_FAMILY_RAG3_TEST_RESULTS)
    if (
        len(qwen_family_rag3_test_df) != len(QWEN_FAMILY_RAG3_TEST_RUNS)
        or not qwen_family_rag3_test_df['status'].eq('completed').all()
    ):
        raise RuntimeError('两个 Qwen 的 RAG3 完整 test 未全部成功，不保存正式 summary。')
    QWEN_FAMILY_RAG3_TEST_SUMMARY_PATH.parent.mkdir(parents=True, exist_ok=True)
    qwen_family_rag3_test_df.to_csv(
        QWEN_FAMILY_RAG3_TEST_SUMMARY_PATH,
        index=False,
        encoding='utf-8-sig',
    )
    LOGGER.info(
        'Qwen 家族 RAG3 test summary 已保存：%s',
        QWEN_FAMILY_RAG3_TEST_SUMMARY_PATH,
    )
    display(qwen_family_rag3_test_df)


In [ ]:
# ===== Qwen3.6 27B No Thinking RAG3 专用补跑：每 100 条重载 =====
RUN_QWEN27_RAG3_RELOAD100_RERUN = False  # 已终止的完整补跑；保留历史配置但不再自动运行。
QWEN27_RAG3_RELOAD100_SUMMARY_PATH = (
    PROJECT_ROOT
    / REPORT_DIR
    / 'qwen27_fixed_rag_k3_test_reload100_rerun_summary.csv'
)
QWEN27_RAG3_RELOAD100_RERUN = next(
    run.copy()
    for run in QWEN_FAMILY_RAG3_TEST_RUNS
    if run['model_key'] == 'local/qwen3.6-27b-no-thinking'
)
QWEN27_RAG3_RELOAD100_RERUN.update({
    'run_id': 'qwen27_fixed_rag_k3_test_reload100_rerun',
    'reload_every_samples': 100,
    'label': 'Qwen3.6 27B No Thinking CNC test fixed_rag_k3 reload100 rerun',
})
display(pd.DataFrame([QWEN27_RAG3_RELOAD100_RERUN]))

if not RUN_QWEN27_RAG3_RELOAD100_RERUN:
    LOGGER.warning('Qwen 27B reload100 专用补跑未启动。')
else:
    QWEN27_RAG3_RELOAD100_RESULTS = run_formal_batch(
        [QWEN27_RAG3_RELOAD100_RERUN],
        existing_retrievers={
            QWEN27_RAG3_RELOAD100_RERUN['run_id']: QWEN_RAG3_RETRIEVER,
        },
        require_all_smoke=False,
    )
    qwen27_rag3_reload100_df = pd.DataFrame(QWEN27_RAG3_RELOAD100_RESULTS)
    if (
        len(qwen27_rag3_reload100_df) != 1
        or qwen27_rag3_reload100_df.iloc[0]['status'] != 'completed'
        or int(qwen27_rag3_reload100_df.iloc[0]['reload_every_samples']) != 100
    ):
        raise RuntimeError('Qwen 27B reload100 专用补跑未成功，不保存正式 summary。')
    QWEN27_RAG3_RELOAD100_SUMMARY_PATH.parent.mkdir(
        parents=True,
        exist_ok=True,
    )
    qwen27_rag3_reload100_df.to_csv(
        QWEN27_RAG3_RELOAD100_SUMMARY_PATH,
        index=False,
        encoding='utf-8-sig',
    )
    LOGGER.info(
        'Qwen 27B reload100 专用补跑 summary 已保存：%s',
        QWEN27_RAG3_RELOAD100_SUMMARY_PATH,
    )
    display(qwen27_rag3_reload100_df)


In [ ]:
# ===== Qwen3.6 27B cache_prompt=False 定点诊断：固定 test 第 590–636 条 =====
RUN_QWEN27_CACHE_PROMPT_OFF_DIAGNOSTIC = True
QWEN27_CACHE_PROMPT_OFF_SUMMARY_PATH = (
    PROJECT_ROOT
    / REPORT_DIR
    / 'qwen27_fixed_rag_k3_cache_prompt_off_diagnostic_590_636_summary.csv'
)
QWEN27_CACHE_PROMPT_OFF_RUN = next(
    run.copy()
    for run in QWEN_FAMILY_RAG3_TEST_RUNS
    if run['model_key'] == 'local/qwen3.6-27b-no-thinking'
)
QWEN27_CACHE_PROMPT_OFF_RUN.update({
    'run_id': 'qwen27_fixed_rag_k3_cache_prompt_off_diagnostic_590_636',
    'reload_every_samples': 0,
    'llm_extra_body': {'cache_prompt': False},
    'label': 'Qwen3.6 27B No Thinking cache_prompt off diagnostic 590-636',
})
qwen27_cache_prompt_off_samples = load_dataset(DATASET_NAME, n=None)[589:636]
if len(qwen27_cache_prompt_off_samples) != 47:
    raise RuntimeError('cache_prompt 诊断子集应包含 47 条样本。')
if (
    str(qwen27_cache_prompt_off_samples[5]['id']) != '2677'
    or str(qwen27_cache_prompt_off_samples[41]['id']) != '110'
):
    raise RuntimeError('cache_prompt 诊断子集的两个已知触发位置不匹配。')
display(pd.DataFrame([{
    **QWEN27_CACHE_PROMPT_OFF_RUN,
    'sample_range_1_based': '590-636',
    'sample_count': len(qwen27_cache_prompt_off_samples),
}]))

if not RUN_QWEN27_CACHE_PROMPT_OFF_DIAGNOSTIC:
    LOGGER.warning('Qwen 27B cache_prompt=False 定点诊断未启动。')
else:
    QWEN27_CACHE_PROMPT_OFF_RESULTS = run_formal_batch(
        [QWEN27_CACHE_PROMPT_OFF_RUN],
        existing_retrievers={
            QWEN27_CACHE_PROMPT_OFF_RUN['run_id']: QWEN_RAG3_RETRIEVER,
        },
        samples_override=qwen27_cache_prompt_off_samples,
        require_all_smoke=False,
    )
    qwen27_cache_prompt_off_df = pd.DataFrame(QWEN27_CACHE_PROMPT_OFF_RESULTS)
    if (
        len(qwen27_cache_prompt_off_df) != 1
        or qwen27_cache_prompt_off_df.iloc[0]['status'] != 'completed'
        or bool(qwen27_cache_prompt_off_df.iloc[0]['cache_prompt'])
    ):
        raise RuntimeError('Qwen 27B cache_prompt=False 定点诊断未成功，不保存 summary。')
    QWEN27_CACHE_PROMPT_OFF_SUMMARY_PATH.parent.mkdir(parents=True, exist_ok=True)
    qwen27_cache_prompt_off_df.to_csv(
        QWEN27_CACHE_PROMPT_OFF_SUMMARY_PATH,
        index=False,
        encoding='utf-8-sig',
    )
    LOGGER.info(
        'Qwen 27B cache_prompt=False 定点诊断 summary 已保存：%s',
        QWEN27_CACHE_PROMPT_OFF_SUMMARY_PATH,
    )
    display(qwen27_cache_prompt_off_df)


In [ ]:
# ===== Qwen3.6 27B 正式续跑：复用前 500 条，仅运行 501–1038 =====
RUN_QWEN27_REUSE500_TAIL = True
QWEN27_REUSE500_SUMMARY_PATH = (
    PROJECT_ROOT
    / REPORT_DIR
    / 'qwen27_fixed_rag_k3_test_reuse500_cache_prompt_off_tail_summary.csv'
)

# 来自 2026-08-27 13:29:50 的 progress 500/1038；全部保留为原始计数，避免由三位小数反推。
QWEN27_PREFIX500_COUNTS = {
    'n_samples': 500,
    'n_causal_gold': 266,
    'n_causal_pred': 281,
    'detection_tp': 228,
    'detection_tn': 181,
    'detection_fp': 53,
    'detection_fn': 38,
    'all_n_eval_samples': 500,
    'all_n_gold_triples': 369,
    'all_n_pred_triples': 313,
    'strict_all_tp': 175,
    'strict_all_fp': 138,
    'strict_all_fn': 194,
    'anchor_all_tp': 184,
    'anchor_all_fp': 129,
    'anchor_all_fn': 185,
    'detected_n_eval_samples': 228,
    'detected_n_gold_triples': 325,
    'detected_n_pred_triples': 254,
    'strict_detected_tp': 175,
    'strict_detected_fp': 79,
    'strict_detected_fn': 150,
    'anchor_detected_tp': 184,
    'anchor_detected_fp': 70,
    'anchor_detected_fn': 141,
}

QWEN27_REUSE500_RUN = next(
    run.copy()
    for run in QWEN_FAMILY_RAG3_TEST_RUNS
    if run['model_key'] == 'local/qwen3.6-27b-no-thinking'
)
QWEN27_REUSE500_RUN.update({
    'run_id': 'qwen27_fixed_rag_k3_test_reuse500_cache_prompt_off_tail',
    'reload_every_samples': 100,
    'llm_extra_body': {'cache_prompt': False},
    'label': 'Qwen3.6 27B No Thinking CNC test fixed_rag_k3 samples 501-1038',
})

qwen27_all_test_samples = load_dataset(DATASET_NAME, n=None)
qwen27_prefix500_samples = qwen27_all_test_samples[:500]
qwen27_tail_samples = qwen27_all_test_samples[500:]
if len(qwen27_all_test_samples) != 1038 or len(qwen27_tail_samples) != 538:
    raise RuntimeError('固定 test 顺序或样本数已变化，不能复用原前 500 条计数。')
if (
    str(qwen27_prefix500_samples[-1]['id']) != '30'
    or str(qwen27_tail_samples[0]['id']) != '1076'
    or sum(bool(sample['has_causal']) for sample in qwen27_prefix500_samples) != 266
    or sum(len(sample['relations']) for sample in qwen27_prefix500_samples) != 369
):
    raise RuntimeError('前 500 条数据与已保存 progress 计数不匹配，停止续跑。')
if 'detection_tp' not in _summary_row.__code__.co_consts:
    raise RuntimeError('请先重新运行“正式批量评测辅助函数”代码块，再执行本续跑。')

display(pd.DataFrame([{
    **QWEN27_REUSE500_RUN,
    'reused_sample_range': '1-500',
    'new_sample_range': '501-1038',
    'new_sample_count': len(qwen27_tail_samples),
}]))

def _prf_from_counts(tp: int, fp: int, fn: int) -> dict[str, float]:
    precision = tp / (tp + fp) if tp + fp else 0.0
    recall = tp / (tp + fn) if tp + fn else 0.0
    f1 = 2 * precision * recall / (precision + recall) if precision + recall else 0.0
    return {'precision': precision, 'recall': recall, 'f1': f1}

if not RUN_QWEN27_REUSE500_TAIL:
    LOGGER.warning('Qwen 27B 复用前 500 条的正式续跑未启动。')
else:
    QWEN27_REUSE500_TAIL_RESULTS = run_formal_batch(
        [QWEN27_REUSE500_RUN],
        existing_retrievers={
            QWEN27_REUSE500_RUN['run_id']: QWEN_RAG3_RETRIEVER,
        },
        samples_override=qwen27_tail_samples,
        require_all_smoke=False,
    )
    qwen27_tail_row = QWEN27_REUSE500_TAIL_RESULTS[0]
    if qwen27_tail_row['status'] != 'completed' or bool(qwen27_tail_row['cache_prompt']):
        raise RuntimeError('Qwen 27B 后 538 条未在 cache_prompt=False 下成功完成。')

    missing_count_fields = sorted(set(QWEN27_PREFIX500_COUNTS) - set(qwen27_tail_row))
    if missing_count_fields:
        raise RuntimeError(f'续跑结果缺少原始计数字段：{missing_count_fields}')
    qwen27_combined_counts = {
        key: int(prefix_value) + int(qwen27_tail_row[key])
        for key, prefix_value in QWEN27_PREFIX500_COUNTS.items()
    }
    if qwen27_combined_counts['n_samples'] != len(qwen27_all_test_samples):
        raise RuntimeError('合并后的样本总数不等于完整 test。')
    if qwen27_combined_counts['n_causal_gold'] != sum(
        bool(sample['has_causal']) for sample in qwen27_all_test_samples
    ):
        raise RuntimeError('合并后的 gold causal 总数不一致。')
    if qwen27_combined_counts['all_n_gold_triples'] != sum(
        len(sample['relations']) for sample in qwen27_all_test_samples
    ):
        raise RuntimeError('合并后的 gold triple 总数不一致。')
    if qwen27_combined_counts['detected_n_eval_samples'] != qwen27_combined_counts['detection_tp']:
        raise RuntimeError('合并后的 detected_only 样本数与 Detection TP 不一致。')

    qwen27_detection = _prf_from_counts(
        qwen27_combined_counts['detection_tp'],
        qwen27_combined_counts['detection_fp'],
        qwen27_combined_counts['detection_fn'],
    )
    qwen27_strict_all = _prf_from_counts(
        qwen27_combined_counts['strict_all_tp'],
        qwen27_combined_counts['strict_all_fp'],
        qwen27_combined_counts['strict_all_fn'],
    )
    qwen27_anchor_all = _prf_from_counts(
        qwen27_combined_counts['anchor_all_tp'],
        qwen27_combined_counts['anchor_all_fp'],
        qwen27_combined_counts['anchor_all_fn'],
    )
    qwen27_strict_detected = _prf_from_counts(
        qwen27_combined_counts['strict_detected_tp'],
        qwen27_combined_counts['strict_detected_fp'],
        qwen27_combined_counts['strict_detected_fn'],
    )
    qwen27_anchor_detected = _prf_from_counts(
        qwen27_combined_counts['anchor_detected_tp'],
        qwen27_combined_counts['anchor_detected_fp'],
        qwen27_combined_counts['anchor_detected_fn'],
    )

    qwen27_combined_row = {
        'status': 'completed_composite',
        'model': QWEN27_REUSE500_RUN['display_name'],
        'model_key': QWEN27_REUSE500_RUN['model_key'],
        'prompt': QWEN27_REUSE500_RUN['prompt_name'],
        'rag': f"{QWEN27_REUSE500_RUN['rag_mode']}-k{QWEN27_REUSE500_RUN['rag_top_k']}",
        'temperature': TEMPERATURE,
        'context_length': int(QWEN27_REUSE500_RUN.get('context_length', CONTEXT_LENGTH)),
        'max_tokens': MAX_TOKENS,
        'parallel': QWEN27_REUSE500_RUN.get('parallel', 'model_default'),
        'reload_every_samples': 100,
        'cache_prompt': 'default samples 1-500; false samples 501-1038',
        'reused_prefix_samples': 500,
        'new_tail_samples': len(qwen27_tail_samples),
        'detection_accuracy': (
            qwen27_combined_counts['detection_tp'] + qwen27_combined_counts['detection_tn']
        ) / qwen27_combined_counts['n_samples'],
        'detection_precision': qwen27_detection['precision'],
        'detection_recall': qwen27_detection['recall'],
        'detection_f1': qwen27_detection['f1'],
        'strict_all_precision': qwen27_strict_all['precision'],
        'strict_all_recall': qwen27_strict_all['recall'],
        'strict_all_f1': qwen27_strict_all['f1'],
        'anchor_all_precision': qwen27_anchor_all['precision'],
        'anchor_all_recall': qwen27_anchor_all['recall'],
        'anchor_all_f1': qwen27_anchor_all['f1'],
        'strict_detected_only_precision': qwen27_strict_detected['precision'],
        'strict_detected_only_recall': qwen27_strict_detected['recall'],
        'strict_detected_only_f1': qwen27_strict_detected['f1'],
        'anchor_detected_only_precision': qwen27_anchor_detected['precision'],
        'anchor_detected_only_recall': qwen27_anchor_detected['recall'],
        'anchor_detected_only_f1': qwen27_anchor_detected['f1'],
        **qwen27_combined_counts,
        'prefix_evidence': 'notebook progress 500/1038 at 2026-08-27 13:29:50',
        'tail_report_path': qwen27_tail_row.get('report_path', ''),
    }
    qwen27_combined_df = pd.DataFrame([qwen27_combined_row])
    QWEN27_REUSE500_SUMMARY_PATH.parent.mkdir(parents=True, exist_ok=True)
    qwen27_combined_df.to_csv(
        QWEN27_REUSE500_SUMMARY_PATH,
        index=False,
        encoding='utf-8-sig',
    )
    LOGGER.info('Qwen 27B 复用前 500 条的完整 test summary 已保存：%s', QWEN27_REUSE500_SUMMARY_PATH)
    display(qwen27_combined_df[[
        'model',
        'detection_precision', 'detection_recall', 'detection_f1',
        'strict_all_precision', 'strict_all_recall', 'strict_all_f1',
        'anchor_all_precision', 'anchor_all_recall', 'anchor_all_f1',
        'strict_detected_only_f1', 'anchor_detected_only_f1',
    ]])
